# ASSIGNMENT 1 - Question 8 - Hourly Electricity-Load Forecasting
### Alumno: Martín Andrés Gutiérrez López
Topics 2D Sequential tensors, Vanilla RNNs, hidden-state transitions, BPTT, time-series forecasting

In [1]:
# ==============================================================================
# Question 8: Hourly Electricity-Load Forecasting (Case: CityGrid Utility)
# Sequence Dataset Shapes, Temporal Split, RNN Hidden States, and BPTT Analysis
# ==============================================================================

import numpy as np
import torch
import torch.nn as nn

BOLD = "\033[1m"
BOLD_BLUE = "\033[1;34m"
RESET = "\033[0m"

# ------------------------------------------------------------------------------
# Part (a): Sample Count and Tensor Shapes
# ------------------------------------------------------------------------------
total_timesteps = 17520
window_size = 24
forecast_horizon = 1
batch_size = 64
num_features = 3

total_samples = total_timesteps - window_size - forecast_horizon + 1
input_batch_shape = (batch_size, window_size, num_features)
target_batch_shape = (batch_size, forecast_horizon)

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (a): DATASET DIMENSIONS AND BATCH SHAPES")
print("=" * 75,f"{RESET}")

print(f"Total built samples (window, target) : {total_samples}")
print(f"Input mini-batch shape (B, T, D)     : {input_batch_shape}")
print(f"Target mini-batch shape (B, H)       : {target_batch_shape}")
print("Dimension meanings:")
print("  - Batch (64)   : Independent 24-hour sequences processed in parallel.")
print("  - Sequence (24): Consecutive hourly time steps in the input window.")
print(
    "  - Features (3) : Hourly features [load (MW), temperature (C),"
    " hour-of-day]."
)
print()

# ------------------------------------------------------------------------------
# Part (b): Train / Validation Chronological Split
# ------------------------------------------------------------------------------
train_ratio = 0.8
train_size = int(total_samples * train_ratio)
val_size = total_samples - train_size

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (b): CHRONOLOGICAL SPLIT VS RANDOM SHUFFLE")
print("=" * 75,f"{RESET}")

print(f"Chronological Train samples (first 80%) : {train_size}")
print(f"Chronological Val samples (last 20%)   : {val_size}")
print(
    "Issue with Random Shuffle              : Consecutive windows overlap by"
    " 23 hours."
)
print(
    "                                         Shuffling causes severe"
    " data leakage."
)
print()

# ------------------------------------------------------------------------------
# Part (c): Step-by-Step Scalar Vanilla RNN Transitions
# ------------------------------------------------------------------------------
w_x = 0.5
w_h = 0.8
b = 0.0
h = 0.0  # h_0
x_sequence = [1.0, 2.0, -1.0]

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (c): HIDDEN STATE TRANSITIONS (h_t = tanh(w_x*x_t + w_h*h_{t-1}))")
print("=" * 75,f"{RESET}")

h_states = []
for t, x_t in enumerate(x_sequence, start=1):
  net = w_x * x_t + w_h * h + b
  h = np.tanh(net)
  h_states.append(h)
  print(f"Step {t}: x_{t} = {x_t:+4.1f} | net_{t} = {net:8.6f} | h_{t} = {h:.6f}")

print(f"\nFinal State vector: h = [{', '.join(f'{val:.6f}' for val in h_states)}]")
print()

# ------------------------------------------------------------------------------
# Part (d): BPTT Gradient Scale Simulation over 49 Steps
# ------------------------------------------------------------------------------
steps = 49
factor_vanishing = 0.5
factor_exploding = 1.5

mag_vanishing = factor_vanishing**steps
mag_exploding = factor_exploding**steps

print(f"\n{BOLD_BLUE}")
print("=" * 75)
print("PART (d): BPTT GRADIENT ESTIMATION OVER 49 TRANSITION STEPS")
print("=" * 75,f"{RESET}")

print(f"Chain rule expression: dL/dh_1 = (dL/dh_50) * Prod_{{t=2}}^{{50}} (dh_t / dh_{{t-1}})")
print(f"When |dh_t/dh_{{t-1}}| ≈ 0.5  -> Magnitude factor: (0.5)^49 ≈ {mag_vanishing:.3e} (Vanishing)")
print(f"When |dh_t/dh_{{t-1}}| ≈ 1.5  -> Magnitude factor: (1.5)^49 ≈ {mag_exploding:.3e} (Exploding)")
print("Connection to Truncated BPTT : Truncating limits the backpropagation window to k steps")
print("                               (e.g., k=24), preventing numerical collapse or explosion.")



PART (a): DATASET DIMENSIONS AND BATCH SHAPES
Total built samples (window, target) : 17496
Input mini-batch shape (B, T, D)     : (64, 24, 3)
Target mini-batch shape (B, H)       : (64, 1)
Dimension meanings:
  - Batch (64)   : Independent 24-hour sequences processed in parallel.
  - Sequence (24): Consecutive hourly time steps in the input window.
  - Features (3) : Hourly features [load (MW), temperature (C), hour-of-day].



PART (b): CHRONOLOGICAL SPLIT VS RANDOM SHUFFLE
Chronological Train samples (first 80%) : 13996
Chronological Val samples (last 20%)   : 3500
Issue with Random Shuffle              : Consecutive windows overlap by 23 hours.
                                         Shuffling causes severe data leakage.



PART (c): HIDDEN STATE TRANSITIONS (h_t = tanh(w_x*x_t + w_h*h_{t-1}))
Step 1: x_1 = +1.0 | net_1 = 0.500000 | h_1 = 0.462117
Step 2: x_2 = +2.0 | net_2 = 1.369694 | h_2 = 0.878622
Step 3: x_3 = -1.0 | net_3 = 0.202898 | h_3 = 0.200159

Final State vector: h =